## Main analysis

### Setup

In [ ]:
# parameters/cut-offs used

# RNA
min_umi = 500
max_umi = 150000
min_genes = 250

knn_n_neighbors = 20
knn_n_pcs = 30

#ATAC
min_frags = 600
min_tsse = 8

# Auxiliary files: 
gene_name_mapping = "references/t2g.txt"



In [ ]:
import snapatac2 as snap
import sys, os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
sys.path.append('./utils')
from importing import download_h5ads, add_metadata_to_adata, build_adata, add_gene_names_to_adata, run_scrublet
from plotting import plot_knee_curve, plot_expression_heatmap, stacked_barplot_proportions
import seaborn as sns
import mpl_scatter_density 
import importlib
import anndata as ad
importlib.reload(sys.modules['importing'])
importlib.reload(sys.modules['plotting'])

os.chdir("/Users/swekhand/Documents/GitHub/share-v2-analysis/")

### Read barcode QC metrics computed in preprocessing steps 

In [ ]:
rna_obs = pd.read_csv("results/filt_rna_100UMI.barcodes_metrics.tsv", sep = "\t")
atac_obs = pd.read_csv("results/filt_atac_100frags.barcode_metrics.tsv", sep = "\t")

rna_adata = sc.read_h5ad("results/filt_rna_100UMI.h5ad", backed=True)



### Fig 4A

In [ ]:
joint = pd.merge(atac_obs, rna_obs, how = "outer", left_on= "barcode", right_on= "cell_barcode")
joint['cell_barcode'] = joint['cell_barcode'].combine_first(joint['barcode'])


In [ ]:
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(1, 1, 1, projection='scatter_density')

density = ax.scatter_density(
    joint["total_counts"],
    joint["n_fragments"],
    norm=PowerNorm(0.3),
    cmap='GnBu'
)

fig.colorbar(density, label="Local Density")
ax.set_xscale("log")
ax.set_yscale("log")
ax.axhline(600, color="gray", linestyle="--")
ax.axvline(500, color="gray", linestyle="--")

ax.set_xlabel("Total UMIs (log10)")
ax.set_ylabel("Number of Unique Fragments (log10)")
sns.despine()
plt.tight_layout()
plt.show()
print("# of barcodes passing thresholds: " + str(len(joint[(joint["total_counts"] >=500) & (joint["n_fragments"] >=600)])))

### Fig 4b

In [ ]:
#TSSe vs num frags plot

sns.kdeplot(data=joint[ (joint["total_counts"] >=500) & (joint["n_fragments"] >=600)], 
            x="n_fragments", y = "tsse", log_scale=[True, False], fill=True)
plt.axhline(linestyle = "--", y=8)

#plt.savefig("figures/Fig2b_updated.png", format="png") 


### Fig 4C

In [ ]:
barcodes = joint.loc[
    (joint["total_counts"] >= 500) & 
    (joint["n_fragments"] >= 600) & 
    (joint["tsse"] >= 8), 
    "cell_barcode"
].tolist()
len(barcodes)

In [ ]:
rna_filt_adata = rna_adata[barcodes, :].to_memory()
rna_filt_adata


In [ ]:
sns.violinplot(data=rna_filt_adata.obs, x="sample", y = "pct_counts_mt",  hue="sample")

#plt.savefig("figures/Fig2c_updated_RNA_MT_violin.png", format="png") 

In [ ]:
sns.violinplot(data=rna_filt_adata.obs, x="sample", y = "total_counts", log_scale=True, hue="sample")

plt.axhline(y=100000, color='grey', linestyle='--', label='high UMI cut-off')
#plt.savefig("figures/Fig2c_updated_RNA_total_UMI_violin.png", format="png") 

### Doublet detection on RNA (Fig 4D and Supp Fig 2)

In [ ]:
sc.pp.scrublet(rna_filt_adata)

In [ ]:
sns.histplot(rna_filt_adata.uns['scrublet'], x="doublet_scores_sim") 
plt.axvline(x=0.1, color='grey', linestyle='--', label='doublet cut-off')

#plt.savefig("figures/SuppFig_simulated_doublets_scores.png", format="png") 

In [ ]:
sns.kdeplot(data=rna_filt_adata.obs, x="doublet_score", hue="sample", fill=True)
plt.axvline(x=0.1, color='grey', linestyle='--', label='doublet cut-off')
#plt.savefig("broad/figures/SuppFig_observed_doublets_scores.png", format="png") 

### Filter ATAC and RNA and cluster (Fig 4E and 4F)

In [ ]:
barcodes = rna_filt_adata.obs_names[rna_filt_adata.obs["doublet_score"] <= 0.1].to_list()
len(barcodes)

In [ ]:
## This UMAP might look different than the one in paper due to randomness in UMAP

rna_filt_adata = rna_filt_adata[rna_filt_adata.obs["doublet_score"] <= 0.1]

### normalize the data ###
sc.pp.normalize_total(rna_filt_adata, target_sum=1e4, layers=None, inplace=True) # Counts per 10k
sc.pp.log1p(rna_filt_adata, layer=None)
gc.collect()
# highly variable genes are used to compute the clustering 
sc.pp.highly_variable_genes(rna_filt_adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

adatas = rna_filt_adata[:, rna_filt_adata.var.highly_variable]
gc.collect()
sc.pp.regress_out(adatas, ['pct_counts_mt','n_genes_by_counts'])
gc.collect()
sc.tl.pca(adatas, svd_solver='arpack')
gc.collect()

sc.pp.neighbors(adatas, n_neighbors=knn_n_neighbors, n_pcs=knn_n_pcs) # put non standard settings near the top
gc.collect()
print("Clustering....")
sc.tl.leiden(adatas, resolution = 1)
sc.tl.umap(adatas, random_state = 0)

rna_filt_adata.uns['neighbors'] = adatas.uns['neighbors']
rna_filt_adata.uns['leiden'] = adatas.uns['leiden']
rna_filt_adata.uns['umap'] = adatas.uns['umap']
rna_filt_adata.obs['leiden'] = adatas.obs['leiden']
rna_filt_adata.obsm = adatas.obsm
rna_filt_adata.obsp = adatas.obsp
gc.collect()

sc.pl.umap(rna_filt_adata, 
           color=['leiden'], 
           size=10, 
           legend_loc = 'on data', 
           legend_fontsize=15,
           show = False
          )

In [ ]:
adataset = snap.read_dataset("data/atac_adataset.h5ad")
filt_atac = adataset.subset(out="data/tmp.h5ad", obs_indices=barcodes)
atac_filt_adata = filt_atac[0].to_adata()


In [ ]:
## warning - compute intensive!!

import pickle 
from pycisTopic.cistopic_class import create_cistopic_object
from pycisTopic.lda_models import evaluate_models
from pycisTopic.clust_vis import (
    find_clusters,
    run_umap,
    run_tsne,
    plot_metadata,
    plot_topic,
    cell_topic_heatmap
)

frag_dict = {'SS-PKR-432': 'data/IGVFFI5731CAQI.bed_sorted.gz',
 'SS-PKR-433': 'data/IGVFFI5042DAMZ.bed_sorted.gz',
 'SS-PKR-434': 'data/IGVFFI0753YWZB.bed_sorted.gz',
 'SS-PKR-435': 'data/IGVFFI6974VPAB.bed_sorted.gz',
 'SS-PKR-436': 'data/IGVFFI5731CAQI.bed_sorted.gz'}

pycis = create_cistopic_object(atac_filt_adata.X.T.tocsr(), 
                               cell_names=atac_filt_adata.obs_names.to_list(), 
                               region_names=atac_filt_adata.var_names.to_list(), 
                               tag_cells=False,
                               path_to_fragments=frag_dict)

In [ ]:
## warning - compute intensive!!
## folloe instructions on installing and running pycistopic model 
## https://pycistopic.readthedocs.io/en/latest/notebooks/human_cerebellum.html#Run-models
os.environ['MALLET_MEMORY'] = '80G'
from pycisTopic.lda_models import run_cgs_models_mallet

mallet_path="/home//Mallet-202108/bin/mallet"

models = run_cgs_models_mallet(cistopic_obj=pycis, 
                               mallet_path=mallet_path,
                               tmp_path="tmp/cistopic-models",
                               n_topics=[5,10,50,75,100],
                               save_path="tmp/cistopic-models",
                               eta=0.1, 
                               eta_by_topic=False,
                               n_iter=500,
                               random_state=555, 
                               alpha=50,alpha_by_topic=True,
                               n_cpu=10
                    )


pickle.dump(
    models,
    open("tmp/models-SHARE-filtered.pkl", "wb")
)

In [ ]:
model = evaluate_models(
    models,
    select_model = 50,
    return_model = True, 
    save="figures/SuppFig_cistopic_model_evaluation.png"
)

In [ ]:
pycis.add_LDA_model(model)
pickle.dump(
    pycis,
    open("tmp/pycis-with-model.pkl", "wb")
)

In [ ]:
find_clusters(
    pycis,
    target  = 'cell',
    k = 10,
    res = [0.5],
    prefix = '',
    scale = True
)

run_umap(
    pycis,
    target  = 'cell', 
    scale=True)

In [ ]:
pycis.cell_data.to_csv("tmp/cistopic-filtered.obs.tsv", sep="\t")
pycis.projections["cell"]["UMAP"].to_csv("tmp/cistopic-filtered.umap.coords.tsv", sep="\t")
pycis.selected_model.cell_topic.to_csv("tmp/cistopic-filtered.50.celltopics.tsv", sep="\t")


In [ ]:
atac_filt_adata = ad.AnnData(X=pycis.selected_model.cell_topic.T)
sc.pp.neighbors(atac_filt_adata, metric='cosine', n_neighbors=25)
sc.tl.leiden(atac_filt_adata,resolution=0.4)
sc.tl.umap(atac_filt_adata, min_dist=0.1, spread=1)
sc.pl.umap(atac_filt_adata, color="leiden")

### Smoothed gene expression (Fig 4G and Supp Fig 3)

In [ ]:
# using ATAC nearest neighbors, calculate smoothed gene expression values. the smoothed gene expression value of a gene in a cell is 
# equal to the average RNA expression across k nearest ATAC neighbors of the cell.

from scipy.sparse import issparse, lil_matrix

def atac_smooth_gene_expression(gene, k=25, sigma=1.0):
    if gene not in rna_filt_adata.var_names:
        gene_idx = rna_filt_adata.var["gene_name_unique"] == gene
        gene = rna_filt_adata.var_names[gene_idx][0]

    expr_series = pd.Series(0.0, index=atac_filt_adata.obs_names, dtype=float)
    common_barcodes = atac_filt_adata.obs_names.intersection(rna_filt_adata.obs_names)

    gene_expr = rna_filt_adata[common_barcodes, gene].X
    if hasattr(gene_expr, "toarray"):
        gene_expr = gene_expr.toarray().flatten()
    expr_series.loc[common_barcodes] = gene_expr

    distances = atac_filt_adata.obsp["distances"]
    n_cells = distances.shape[0]

    if not isinstance(distances, lil_matrix):
        distances = distances.tolil()

    for i in range(n_cells):
        row = distances.rows[i]
        data = distances.data[i]
        if len(data) > k:
            top_k_idx = np.argsort(data)[:k]
            distances.rows[i] = [row[j] for j in top_k_idx]
            distances.data[i] = [data[j] for j in top_k_idx]

    # Apply Gaussian kernel: similarity = exp(-d^2 / (2 * sigma^2))
    for i in range(n_cells):
        distances.data[i] = np.exp(-np.square(distances.data[i]) / (2 * sigma**2)).tolist()

    similarities = distances.tocsr()

    # Normalize rows
    row_sums = np.array(similarities.sum(axis=1)).flatten()
    row_sums[row_sums == 0] = 1
    row_norm = similarities.multiply(1 / row_sums[:, np.newaxis])

    # Smooth the expression
    smoothed_expr = row_norm.dot(expr_series.values)

    return smoothed_expr

In [ ]:
# using RNA nearest neighbors, calculate smoothed gene expression values. the smoothed gene expression value of a gene in a cell is 
# equal to the average RNA expression across k nearest RNA neighbors of the cell.

from scipy.sparse import csr_matrix

def rna_smooth_gene_expression(gene, top_k=25):
    if gene not in rna_filt_adata.var_names:
        gene_idx = rna_filt_adata.var["gene_name_unique"] == gene
        gene = rna_filt_adata.var_names[gene_idx][0]
    
    gene_expr = rna_filt_adata[:, gene].X
    if hasattr(gene_expr, "toarray"):
        gene_expr = gene_expr.toarray().flatten()

    connectivities = rna_filt_adata.obsp["connectivities"].tocsr()

    # Keep only top_k neighbors per row
    top_k_data = []
    top_k_indices = []
    indptr = [0]

    for row_start, row_end in zip(connectivities.indptr[:-1], connectivities.indptr[1:]):
        row_data = connectivities.data[row_start:row_end]
        row_indices = connectivities.indices[row_start:row_end]
        
        if len(row_data) > top_k:
            top_k_idx = np.argsort(row_data)[-top_k:]
            filtered_data = row_data[top_k_idx]
            filtered_indices = row_indices[top_k_idx]
        else:
            filtered_data = row_data
            filtered_indices = row_indices
        
        top_k_data.extend(filtered_data)
        top_k_indices.extend(filtered_indices)
        indptr.append(len(top_k_data))

    # Build sparse matrix with top_k neighbors
    filtered_connectivities = csr_matrix(
        (top_k_data, top_k_indices, indptr), shape=connectivities.shape
    )
    
    row_sums = np.array(filtered_connectivities.sum(axis=1)).flatten()
    row_sums[row_sums == 0] = 1  # avoid division by 0
    row_norm = filtered_connectivities.multiply(1 / row_sums[:, np.newaxis])

    # Smooth expression
    smoothed_expr = row_norm.dot(gene_expr)

    return smoothed_expr


In [ ]:
gene = "Trem2"
atac_filt_adata.obs["smoothed_expr"] = atac_smooth_gene_expression(gene, sigma=0.5)
print(sc.pl.umap(atac_filt_adata, color="smoothed_expr", color_map=("PuBu"), title=gene))

rna_filt_adata.obs["smoothed_expr"] = smooth_gene_expression(gene, top_k=10)
print(sc.pl.umap(rna_filt_adata, color="smoothed_expr", color_map=("PuBu"), title=gene ))

In [ ]:
filt_atac_obs_indexed = filt_atac_obs.set_index("barcode")
filt_atac_obs_indexed["cell_barcode"] = filt_atac_obs_indexed["sample"] + ":" + filt_atac_obs_indexed.index
#filt_atac_obs_indexed

selected_barcodes = filt_atac_obs_indexed.loc[
    filt_atac_obs_indexed["doublet_probability"] <= 0.7, "cell_barcode"
].tolist()

#### WARNING: this will modify the ATAC h5ad on disk!! Advise making a copy of this file before continuing this step.


In [ ]:
atac_adata.subset(obs_indices=selected_barcodes)

In [ ]:
snap.tl.spectral(atac_adata)
snap.tl.umap(atac_adata)
snap.pp.knn(atac_adata)
snap.tl.leiden(atac_adata)

In [ ]:
snap.pl.umap(atac_adata, color='leiden', interactive=False, out_file = "figures/atac_umap.svg")


In [ ]:
atac_adata.write("results/SHAREv2-ATAC-final.h5ad")

In [ ]:
adata.write_h5ad("results/SHAREv2-RNA-final.h5ad")